# Workflow Planning Evaluator - Smoke Test

In [5]:
import json, os, random, glob, importlib.util
from dotenv import load_dotenv

load_dotenv()

from _utils import format_workflow_trace_for_eval

DATA_DIR = r"C:\Users\selshafey\Downloads\agent-framework\python\samples\getting_started\workflows\orchestration\test_scenarios\converted_data"
all_files = glob.glob(os.path.join(DATA_DIR, "**", "*.json"), recursive=True)
sample_file = random.choice(all_files)
print(f"File: {os.path.relpath(sample_file, DATA_DIR)}")

with open(sample_file, encoding="utf-8") as f:
    data = json.load(f)

print(f"workflow_type: {data.get('workflow_type')}, agents: {list(data.get('agents', {}).keys())}")
print(f"invocations: {len(data.get('invocations', []))}, errors: {len(data.get('errors', []))}")

File: magentic\multi_interview_scheduling_SCHED-001_20260216_215817.json
workflow_type: magentic, agents: ['ReqMaster', 'Scheduler']
invocations: 8, errors: 0


In [6]:
text = format_workflow_trace_for_eval(data)
print(text)

truncated: some messages were truncated, evaluation quality may be reduced


[Workflow Definition]
Executors: magentic_orchestrator (MagenticOrchestrator), ReqMaster (MagenticAgentExecutor), Scheduler (MagenticAgentExecutor), CommsBot (MagenticAgentExecutor)
Edges:
  magentic_orchestrator -> ReqMaster
  ReqMaster -> magentic_orchestrator
  magentic_orchestrator -> Scheduler
  Scheduler -> magentic_orchestrator
  magentic_orchestrator -> CommsBot
  CommsBot -> magentic_orchestrator
Max iterations: 100

----------------------------------------

[User Input]
Schedule an interview for candidate CAND-EXT-001 with hiring manager EMP-HM-001 for the Senior Software Engineer position.

----------------------------------------

[ReqMaster - Invocation 1]
  [System Prompt]
  You are a Requisition Master agent handling job intake and requirements analysis.

Your responsibilities:
1. Retrieve and validate job posting details
2. Extract structured requirements (must-have vs nice-to-have)
3. Verify job is approved and open for sourcing
4. Identify hiring manager and team cont

In [8]:
from azure.ai.evaluation._evaluators._workflow_planning import _WorkflowPlanningEvaluator

model_config = {
    "azure_endpoint": os.environ["EVAL_GPT4_1_ENDPOINT"],
    "azure_deployment": os.environ["EVAL_GPT4_1_DEPLOYMENT_NAME"],
    "api_version": os.environ.get("EVAL_GPT4_1_API_VERSION", "2024-12-01-preview"),
}

evaluator = _WorkflowPlanningEvaluator(model_config=model_config)
result = evaluator(workflow_trace=data)

result

truncated: some messages were truncated, evaluation quality may be reduced


{'workflow_planning': 1,
 'workflow_planning_result': 'pass',
 'workflow_planning_reason': 'The workflow was well-planned. The orchestrator dynamically decomposed the scheduling request, routed tasks to the appropriate agents, tracked progress, and ensured completion. No errors occurred, and all steps were logically sequenced and completed.',
 'workflow_planning_details': {'task_decomposition': 'The request was decomposed into job validation, availability checking, slot selection, and communication, each mapped to a specialized agent.',
  'agent_selection': 'ReqMaster, Scheduler, and CommsBot were invoked for tasks matching their expertise, in a logical and efficient order.',
  'progress_tracking': 'Progress was explicitly tracked with JSON status checks, and the orchestrator advanced steps only after prior completion.',
  'error_handling': 'No errors or tool failures occurred; all tool calls succeeded and results were handled appropriately.'},
 'workflow_planning_prompt_tokens': 10397